In [20]:
import databay
from pyspark.sql import Row
from databay.runtime import DockConfig, dock_spark_init

In [21]:
cfg = DockConfig(
    image="spark-pg-iceberg",
    name="spark-iceberg",
    ports={"4040/tcp": 4040, "15002/tcp": 15002},
    named_volumes={
        "iceberg-lakehouse": "/lakehouse",
        "iceberg-metastore": "/metastore/pgdata"
    },
    bind_mounts={
        "C:\landing": "/data/apache"
    }
)

In [22]:
spark = dock_spark_init(cfg=cfg)

Container started: d42fc5ec1e27
Stopping existing Spark session...
Spark 4.0.1 | Cell magics registered: %%ts, %%skip, %%sparksql


# UNIT Testing - compare_datasets

In [23]:
# Test 1: Identical datasets - 100% match
print("\n[TEST 1] Identical datasets - 100% match")
data_identical = [
    Row(id=1, name="Alice", age=25, city="New York"),
    Row(id=2, name="Bob", age=30, city="London"),
    Row(id=3, name="Charlie", age=35, city="Paris")
]

df_test1_a = spark.createDataFrame(data_identical)
df_test1_b = spark.createDataFrame(data_identical)

result1 = databay.compare_datasets(df_test1_a, df_test1_b, cols=["*"], name_a="DatasetA", name_b="DatasetB")
result1.show(truncate=False)

# Assertions
result1_dict = {row["metric"] + "_" + row["scope"]: row for row in result1.collect()}
assert result1_dict["row_count_DatasetA"]["count_value"] == 3
assert result1_dict["row_count_DatasetB"]["count_value"] == 3
assert result1_dict["diff_rows_DatasetA"]["count_value"] == 0
assert result1_dict["diff_rows_DatasetB"]["count_value"] == 0
assert result1_dict["common_rows_both"]["count_value"] == 3
assert result1_dict["jaccard_pct_both"]["percent_value"] == 100.0
print("✓ TEST 1 PASSED: Identical datasets detected correctly")


[TEST 1] Identical datasets - 100% match
+--------------+-----------------+-----------+-------------+
|metric        |scope            |count_value|percent_value|
+--------------+-----------------+-----------+-------------+
|row_count     |DatasetA         |3          |NULL         |
|row_count     |DatasetB         |3          |NULL         |
|diff_rows     |DatasetA         |0          |NULL         |
|diff_rows     |DatasetB         |0          |NULL         |
|common_rows   |both             |3          |NULL         |
|union_rows    |both             |3          |NULL         |
|match_pct     |DatasetA→DatasetB|3          |100.0        |
|match_pct     |DatasetB→DatasetA|3          |100.0        |
|jaccard_pct   |both             |3          |100.0        |
|size_ratio_pct|DatasetA→DatasetB|3          |100.0        |
+--------------+-----------------+-----------+-------------+

✓ TEST 1 PASSED: Identical datasets detected correctly


In [24]:
# Test 2: Partially matching datasets
print("\n[TEST 2] Partially matching datasets with some differences")
data_partial_a = [
    Row(id=1, name="Alice", age=25, city="New York"),
    Row(id=2, name="Bob", age=30, city="London"),
    Row(id=3, name="Charlie", age=35, city="Paris"),
    Row(id=4, name="David", age=28, city="Berlin"),
    Row(id=5, name="Eve", age=32, city="Madrid")
]

data_partial_b = [
    Row(id=1, name="Alice", age=25, city="New York"),      # match
    Row(id=2, name="Bob", age=31, city="London"),          # age changed
    Row(id=3, name="Charlie", age=35, city="Paris"),       # match
    Row(id=4, name="David", age=28, city="Munich"),        # city changed
    Row(id=6, name="Frank", age=29, city="Rome")           # new row
]

df_test2_a = spark.createDataFrame(data_partial_a)
df_test2_b = spark.createDataFrame(data_partial_b)

result2 = databay.compare_datasets(df_test2_a, df_test2_b, cols=["*"])
result2.show(truncate=False)

# Assertions
result2_dict = {row["metric"] + "_" + row["scope"]: row for row in result2.collect()}
assert result2_dict["row_count_Dataset A"]["count_value"] == 5
assert result2_dict["row_count_Dataset B"]["count_value"] == 5
assert result2_dict["common_rows_both"]["count_value"] == 2  # Only Alice and Charlie match
assert result2_dict["union_rows_both"]["count_value"] == 8   # 5 + 5 - 2 = 8
jaccard = result2_dict["jaccard_pct_both"]["percent_value"]
assert 24 <= jaccard <= 26, f"Expected Jaccard ~25%, got {jaccard}"  # 2/8 = 25%
print("✓ TEST 2 PASSED: Partial matches calculated correctly")


[TEST 2] Partially matching datasets with some differences
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |5          |NULL         |
|row_count     |Dataset B          |5          |NULL         |
|diff_rows     |Dataset A          |3          |NULL         |
|diff_rows     |Dataset B          |3          |NULL         |
|common_rows   |both               |2          |NULL         |
|union_rows    |both               |8          |NULL         |
|match_pct     |Dataset A→Dataset B|2          |40.0         |
|match_pct     |Dataset B→Dataset A|2          |40.0         |
|jaccard_pct   |both               |2          |25.0         |
|size_ratio_pct|Dataset A→Dataset B|5          |100.0        |
+--------------+-------------------+-----------+-------------+

✓ TEST 2 PASSED: Partial matches calculated correctly


In [25]:
# Test 3: Completely different datasets - no matches
print("\n[TEST 3] Completely different datasets - no matches")
data_diff_a = [
    Row(id=1, name="Alice", age=25),
    Row(id=2, name="Bob", age=30),
    Row(id=3, name="Charlie", age=35)
]

data_diff_b = [
    Row(id=10, name="Xavier", age=40),
    Row(id=11, name="Yara", age=45),
    Row(id=12, name="Zoe", age=50)
]

df_test3_a = spark.createDataFrame(data_diff_a)
df_test3_b = spark.createDataFrame(data_diff_b)

result3 = databay.compare_datasets(df_test3_a, df_test3_b, cols=["*"])
result3.show(truncate=False)

# Assertions
result3_dict = {row["metric"] + "_" + row["scope"]: row for row in result3.collect()}
assert result3_dict["common_rows_both"]["count_value"] == 0
assert result3_dict["union_rows_both"]["count_value"] == 6  # 3 + 3 - 0 = 6
assert result3_dict["jaccard_pct_both"]["percent_value"] == 0.0
print("✓ TEST 3 PASSED: No matches detected correctly")


[TEST 3] Completely different datasets - no matches
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |3          |NULL         |
|row_count     |Dataset B          |3          |NULL         |
|diff_rows     |Dataset A          |3          |NULL         |
|diff_rows     |Dataset B          |3          |NULL         |
|common_rows   |both               |0          |NULL         |
|union_rows    |both               |6          |NULL         |
|match_pct     |Dataset A→Dataset B|0          |0.0          |
|match_pct     |Dataset B→Dataset A|0          |0.0          |
|jaccard_pct   |both               |0          |0.0          |
|size_ratio_pct|Dataset A→Dataset B|3          |100.0        |
+--------------+-------------------+-----------+-------------+

✓ TEST 3 PASSED: No matches detected correctly


In [26]:
# Test 4: Subset relationship - dataset B is subset of dataset A
print("\n[TEST 4] Subset relationship - B is subset of A")
data_superset = [
    Row(id=1, name="Alice", age=25),
    Row(id=2, name="Bob", age=30),
    Row(id=3, name="Charlie", age=35),
    Row(id=4, name="David", age=28),
    Row(id=5, name="Eve", age=32)
]

data_subset = [
    Row(id=1, name="Alice", age=25),
    Row(id=2, name="Bob", age=30)
]

df_test4_a = spark.createDataFrame(data_superset)
df_test4_b = spark.createDataFrame(data_subset)

result4 = databay.compare_datasets(df_test4_a, df_test4_b, cols=["*"], name_a="Superset", name_b="Subset")
result4.show(truncate=False)

# Assertions
result4_dict = {row["metric"] + "_" + row["scope"]: row for row in result4.collect()}
assert result4_dict["row_count_Superset"]["count_value"] == 5
assert result4_dict["row_count_Subset"]["count_value"] == 2
assert result4_dict["common_rows_both"]["count_value"] == 2
assert result4_dict["diff_rows_Superset"]["count_value"] == 3  # 3 rows not in subset
assert result4_dict["diff_rows_Subset"]["count_value"] == 0    # all rows in superset
match_pct_b_to_a = result4_dict["match_pct_Subset→Superset"]["percent_value"]
assert match_pct_b_to_a == 100.0, "All Subset rows should be in Superset"
print("✓ TEST 4 PASSED: Subset relationship detected correctly")


[TEST 4] Subset relationship - B is subset of A
+--------------+---------------+-----------+-------------+
|metric        |scope          |count_value|percent_value|
+--------------+---------------+-----------+-------------+
|row_count     |Superset       |5          |NULL         |
|row_count     |Subset         |2          |NULL         |
|diff_rows     |Superset       |3          |NULL         |
|diff_rows     |Subset         |0          |NULL         |
|common_rows   |both           |2          |NULL         |
|union_rows    |both           |5          |NULL         |
|match_pct     |Superset→Subset|2          |40.0         |
|match_pct     |Subset→Superset|2          |100.0        |
|jaccard_pct   |both           |2          |40.0         |
|size_ratio_pct|Superset→Subset|5          |250.0        |
+--------------+---------------+-----------+-------------+

✓ TEST 4 PASSED: Subset relationship detected correctly


In [27]:
# Test 5: Specific columns comparison
print("\n[TEST 5] Comparing specific columns only")
data_cols_a = [
    Row(id=1, name="Alice", age=25, city="New York"),
    Row(id=2, name="Bob", age=30, city="London")
]

data_cols_b = [
    Row(id=1, name="Alice", age=99, city="Tokyo"),      # different age & city
    Row(id=2, name="Bob", age=88, city="Paris")         # different age & city
]

df_test5_a = spark.createDataFrame(data_cols_a)
df_test5_b = spark.createDataFrame(data_cols_b)

# Compare only id and name columns - should match
result5 = databay.compare_datasets(df_test5_a, df_test5_b, cols=["id", "name"])
result5.show(truncate=False)

# Assertions
result5_dict = {row["metric"] + "_" + row["scope"]: row for row in result5.collect()}
assert result5_dict["common_rows_both"]["count_value"] == 2, "id and name match in both rows"
assert result5_dict["jaccard_pct_both"]["percent_value"] == 100.0
print("✓ TEST 5 PASSED: Specific column comparison works correctly")


[TEST 5] Comparing specific columns only
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |2          |NULL         |
|row_count     |Dataset B          |2          |NULL         |
|diff_rows     |Dataset A          |0          |NULL         |
|diff_rows     |Dataset B          |0          |NULL         |
|common_rows   |both               |2          |NULL         |
|union_rows    |both               |2          |NULL         |
|match_pct     |Dataset A→Dataset B|2          |100.0        |
|match_pct     |Dataset B→Dataset A|2          |100.0        |
|jaccard_pct   |both               |2          |100.0        |
|size_ratio_pct|Dataset A→Dataset B|2          |100.0        |
+--------------+-------------------+-----------+-------------+

✓ TEST 5 PASSED: Specific column comparison works correctly


In [28]:
# Test 6: Empty dataset comparison
print("\n[TEST 6] One empty dataset")
data_nonempty = [
    Row(id=1, name="Alice"),
    Row(id=2, name="Bob")
]
data_empty = []

df_test6_a = spark.createDataFrame(data_nonempty)
df_test6_b = spark.createDataFrame(data_empty, df_test6_a.schema)

result6 = databay.compare_datasets(df_test6_a, df_test6_b, cols=["*"])
result6.show(truncate=False)

# Assertions
result6_dict = {row["metric"] + "_" + row["scope"]: row for row in result6.collect()}
assert result6_dict["row_count_Dataset A"]["count_value"] == 2
assert result6_dict["row_count_Dataset B"]["count_value"] == 0
assert result6_dict["common_rows_both"]["count_value"] == 0
assert result6_dict["jaccard_pct_both"]["percent_value"] == 0.0
print("✓ TEST 6 PASSED: Empty dataset handled correctly")


[TEST 6] One empty dataset
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |2          |NULL         |
|row_count     |Dataset B          |0          |NULL         |
|diff_rows     |Dataset A          |2          |NULL         |
|diff_rows     |Dataset B          |0          |NULL         |
|common_rows   |both               |0          |NULL         |
|union_rows    |both               |2          |NULL         |
|match_pct     |Dataset A→Dataset B|0          |0.0          |
|match_pct     |Dataset B→Dataset A|0          |0.0          |
|jaccard_pct   |both               |0          |0.0          |
|size_ratio_pct|Dataset A→Dataset B|2          |0.0          |
+--------------+-------------------+-----------+-------------+

✓ TEST 6 PASSED: Empty dataset handled correctly


In [29]:
# Test 7: Size ratio verification
print("\n[TEST 7] Size ratio calculation")
data_small = [Row(id=1, name="A")]
data_large = [Row(id=1, name="A"), Row(id=2, name="B"), Row(id=3, name="C"), Row(id=4, name="D")]

df_test7_a = spark.createDataFrame(data_small)
df_test7_b = spark.createDataFrame(data_large)

result7 = databay.compare_datasets(df_test7_a, df_test7_b, cols=["*"], name_a="Small", name_b="Large")
result7.show(truncate=False)

# Assertions
result7_dict = {row["metric"] + "_" + row["scope"]: row for row in result7.collect()}
size_ratio = result7_dict["size_ratio_pct_Small→Large"]["percent_value"]
assert size_ratio == 25.0, f"Expected 25% (1/4), got {size_ratio}"  # 1/4 = 0.25 = 25%
print("✓ TEST 7 PASSED: Size ratio calculated correctly")


[TEST 7] Size ratio calculation
+--------------+-----------+-----------+-------------+
|metric        |scope      |count_value|percent_value|
+--------------+-----------+-----------+-------------+
|row_count     |Small      |1          |NULL         |
|row_count     |Large      |4          |NULL         |
|diff_rows     |Small      |0          |NULL         |
|diff_rows     |Large      |3          |NULL         |
|common_rows   |both       |1          |NULL         |
|union_rows    |both       |4          |NULL         |
|match_pct     |Small→Large|1          |100.0        |
|match_pct     |Large→Small|1          |25.0         |
|jaccard_pct   |both       |1          |25.0         |
|size_ratio_pct|Small→Large|1          |25.0         |
+--------------+-----------+-----------+-------------+

✓ TEST 7 PASSED: Size ratio calculated correctly


In [30]:
# Test 8: Both datasets empty
print("\n[TEST 8] Both datasets empty")
data_empty_1 = []
data_empty_2 = []

# Need a schema for empty dataframes
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True)
])

df_test8_a = spark.createDataFrame(data_empty_1, schema)
df_test8_b = spark.createDataFrame(data_empty_2, schema)

result8 = databay.compare_datasets(df_test8_a, df_test8_b, cols=["*"])
result8.show(truncate=False)

# Assertions
result8_dict = {row["metric"] + "_" + row["scope"]: row for row in result8.collect()}
assert result8_dict["row_count_Dataset A"]["count_value"] == 0
assert result8_dict["row_count_Dataset B"]["count_value"] == 0
assert result8_dict["common_rows_both"]["count_value"] == 0
assert result8_dict["union_rows_both"]["count_value"] == 0
assert result8_dict["jaccard_pct_both"]["percent_value"] == 0.0
assert result8_dict["match_pct_Dataset A→Dataset B"]["percent_value"] == 0.0
assert result8_dict["match_pct_Dataset B→Dataset A"]["percent_value"] == 0.0
print("✓ TEST 8 PASSED: Both empty datasets handled correctly")


[TEST 8] Both datasets empty
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |0          |NULL         |
|row_count     |Dataset B          |0          |NULL         |
|diff_rows     |Dataset A          |0          |NULL         |
|diff_rows     |Dataset B          |0          |NULL         |
|common_rows   |both               |0          |NULL         |
|union_rows    |both               |0          |NULL         |
|match_pct     |Dataset A→Dataset B|0          |0.0          |
|match_pct     |Dataset B→Dataset A|0          |0.0          |
|jaccard_pct   |both               |0          |0.0          |
|size_ratio_pct|Dataset A→Dataset B|0          |0.0          |
+--------------+-------------------+-----------+-------------+

✓ TEST 8 PASSED: Both empty datasets handled correctly


In [31]:
# Test 9: Duplicate rows handling
print("\n[TEST 9] Datasets with duplicate rows")
data_dupe_a = [
    Row(id=1, name="Alice"),
    Row(id=1, name="Alice"),  # duplicate
    Row(id=2, name="Bob")
]

data_dupe_b = [
    Row(id=1, name="Alice"),
    Row(id=2, name="Bob"),
    Row(id=2, name="Bob")  # duplicate
]

df_test9_a = spark.createDataFrame(data_dupe_a)
df_test9_b = spark.createDataFrame(data_dupe_b)

result9 = databay.compare_datasets(df_test9_a, df_test9_b, cols=["*"])
result9.show(truncate=False)

# Assertions - exceptAll should handle duplicates correctly
result9_dict = {row["metric"] + "_" + row["scope"]: row for row in result9.collect()}
assert result9_dict["row_count_Dataset A"]["count_value"] == 3
assert result9_dict["row_count_Dataset B"]["count_value"] == 3
# A has: [Alice, Alice, Bob], B has: [Alice, Bob, Bob]
# diff_a = A - B = [Alice] (one extra Alice)
# diff_b = B - A = [Bob] (one extra Bob)
assert result9_dict["diff_rows_Dataset A"]["count_value"] == 1
assert result9_dict["diff_rows_Dataset B"]["count_value"] == 1
assert result9_dict["common_rows_both"]["count_value"] == 2  # 3 - 1 = 2
print("✓ TEST 9 PASSED: Duplicate rows handled correctly")


[TEST 9] Datasets with duplicate rows
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |3          |NULL         |
|row_count     |Dataset B          |3          |NULL         |
|diff_rows     |Dataset A          |1          |NULL         |
|diff_rows     |Dataset B          |1          |NULL         |
|common_rows   |both               |2          |NULL         |
|union_rows    |both               |4          |NULL         |
|match_pct     |Dataset A→Dataset B|2          |66.6667      |
|match_pct     |Dataset B→Dataset A|2          |66.6667      |
|jaccard_pct   |both               |2          |50.0         |
|size_ratio_pct|Dataset A→Dataset B|3          |100.0        |
+--------------+-------------------+-----------+-------------+

✓ TEST 9 PASSED: Duplicate rows handled correctly


In [32]:
# Test 10: NULL values in data
print("\n[TEST 10] Datasets with NULL values")
data_null_a = [
    Row(id=1, name="Alice", age=25),
    Row(id=2, name="Bob", age=None),    # NULL age
    Row(id=3, name=None, age=35)        # NULL name
]

data_null_b = [
    Row(id=1, name="Alice", age=25),
    Row(id=2, name="Bob", age=None),    # NULL age - same as A
    Row(id=3, name=None, age=40)        # NULL name but different age
]

df_test10_a = spark.createDataFrame(data_null_a)
df_test10_b = spark.createDataFrame(data_null_b)

result10 = databay.compare_datasets(df_test10_a, df_test10_b, cols=["*"])
result10.show(truncate=False)

# Assertions
result10_dict = {row["metric"] + "_" + row["scope"]: row for row in result10.collect()}
assert result10_dict["row_count_Dataset A"]["count_value"] == 3
assert result10_dict["row_count_Dataset B"]["count_value"] == 3
# Row 1 matches exactly, Row 2 matches exactly (including NULL)
# Row 3 differs (different age values)
assert result10_dict["common_rows_both"]["count_value"] == 2
assert result10_dict["diff_rows_Dataset A"]["count_value"] == 1
assert result10_dict["diff_rows_Dataset B"]["count_value"] == 1
print("✓ TEST 10 PASSED: NULL values handled correctly")


[TEST 10] Datasets with NULL values
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |3          |NULL         |
|row_count     |Dataset B          |3          |NULL         |
|diff_rows     |Dataset A          |1          |NULL         |
|diff_rows     |Dataset B          |1          |NULL         |
|common_rows   |both               |2          |NULL         |
|union_rows    |both               |4          |NULL         |
|match_pct     |Dataset A→Dataset B|2          |66.6667      |
|match_pct     |Dataset B→Dataset A|2          |66.6667      |
|jaccard_pct   |both               |2          |50.0         |
|size_ratio_pct|Dataset A→Dataset B|3          |100.0        |
+--------------+-------------------+-----------+-------------+

✓ TEST 10 PASSED: NULL values handled correctly


In [33]:
# Test 11: Column order differences with cols=["*"]
print("\n[TEST 11] Same data but different column order")
data_order_a = [
    Row(id=1, name="Alice", age=25),
    Row(id=2, name="Bob", age=30)
]

# Create with different column order
df_test11_a = spark.createDataFrame(data_order_a)  # id, name, age
df_test11_b = spark.createDataFrame(data_order_a).select("age", "id", "name")  # age, id, name

result11 = databay.compare_datasets(df_test11_a, df_test11_b, cols=["*"])
result11.show(truncate=False)

# Assertions - should still match because cols=["*"] now normalizes column order
result11_dict = {row["metric"] + "_" + row["scope"]: row for row in result11.collect()}
assert result11_dict["common_rows_both"]["count_value"] == 2
assert result11_dict["jaccard_pct_both"]["percent_value"] == 100.0
print("✓ TEST 11 PASSED: Column order handled correctly with cols=['*']")


[TEST 11] Same data but different column order
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |2          |NULL         |
|row_count     |Dataset B          |2          |NULL         |
|diff_rows     |Dataset A          |0          |NULL         |
|diff_rows     |Dataset B          |0          |NULL         |
|common_rows   |both               |2          |NULL         |
|union_rows    |both               |2          |NULL         |
|match_pct     |Dataset A→Dataset B|2          |100.0        |
|match_pct     |Dataset B→Dataset A|2          |100.0        |
|jaccard_pct   |both               |2          |100.0        |
|size_ratio_pct|Dataset A→Dataset B|2          |100.0        |
+--------------+-------------------+-----------+-------------+

✓ TEST 11 PASSED: Column order handled correctly with cols=['*']


In [34]:
# Test 12: Non-existent column in cols parameter
print("\n[TEST 12] Column that doesn't exist - error handling")
data_test12 = [Row(id=1, name="Alice")]

df_test12_a = spark.createDataFrame(data_test12)
df_test12_b = spark.createDataFrame(data_test12)

try:
    # Try to compare with a column that doesn't exist
    result12 = databay.compare_datasets(df_test12_a, df_test12_b, cols=["id", "nonexistent_column"])
    result12.show()
    print("✗ TEST 12 FAILED: Should have raised an error for non-existent column")
except Exception as e:
    print(f"✓ TEST 12 PASSED: Correctly raised error: {type(e).__name__}")

{"ts": "2026-02-23 00:39:08.051", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `nonexistent_column` cannot be resolved. Did you mean one of the following? [`name`, `id`]. SQLSTATE: 42703;\n'Project [id#20984L, 'nonexistent_column]\n+- LocalRelation [id#20984L, name#20985]\n\n\nJVM stacktrace:\norg.apache.spark.sql.catalyst.ExtendedAnalysisException\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:401)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.org$apache$spark$sql$catalyst$analysis$CheckAnalysis$$failUnresolvedAttribute(CheckAnalysis.scala:169)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$7(CheckAnalysis.scala:404)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$7$adapted(CheckAnalysis.scala:402)\n\tat org.apache.spark.sql.catal


[TEST 12] Column that doesn't exist - error handling
✓ TEST 12 PASSED: Correctly raised error: AnalysisException


In [35]:
# Test 13: Different schemas with cols=["*"]
print("\n[TEST 13] Different schemas - cols=['*'] should select from df_a")
data_schema_a = [
    Row(id=1, name="Alice", age=25)
]

data_schema_b = [
    Row(id=1, name="Alice", city="NYC")  # 'city' instead of 'age'
]

df_test13_a = spark.createDataFrame(data_schema_a)
df_test13_b = spark.createDataFrame(data_schema_b)

try:
    # This should fail because schemas don't match
    result13 = databay.compare_datasets(df_test13_a, df_test13_b, cols=["*"])
    result13.show()
    print("✗ TEST 13 FAILED: Should have raised an error for schema mismatch")
except Exception as e:
    print(f"✓ TEST 13 PASSED: Correctly raised error for schema mismatch: {type(e).__name__}")

{"ts": "2026-02-23 00:39:08.116", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `age` cannot be resolved. Did you mean one of the following? [`id`, `name`, `city`]. SQLSTATE: 42703;\n'Project [id#21010L, name#21011, 'age]\n+- LocalRelation [id#21010L, name#21011, city#21012]\n\n\nJVM stacktrace:\norg.apache.spark.sql.catalyst.ExtendedAnalysisException\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:401)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.org$apache$spark$sql$catalyst$analysis$CheckAnalysis$$failUnresolvedAttribute(CheckAnalysis.scala:169)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$7(CheckAnalysis.scala:404)\n\tat org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$7$adapted(CheckAnalysis.scala:402)\n\tat org.apache.spark.sql.cat


[TEST 13] Different schemas - cols=['*'] should select from df_a
✓ TEST 13 PASSED: Correctly raised error for schema mismatch: AnalysisException


In [36]:
# Test 14: Large match/diff percentages verify decimal precision
print("\n[TEST 14] Decimal precision in percentage calculations")
# Create dataset where 99 out of 100 rows match
data_precision = [Row(id=i, name=f"Person{i}") for i in range(1, 100)]
data_precision_a = data_precision + [Row(id=100, name="UniqueInA")]
data_precision_b = data_precision + [Row(id=101, name="UniqueInB")]

df_test14_a = spark.createDataFrame(data_precision_a)
df_test14_b = spark.createDataFrame(data_precision_b)

result14 = databay.compare_datasets(df_test14_a, df_test14_b, cols=["*"])
result14.show(truncate=False)

# Assertions
result14_dict = {row["metric"] + "_" + row["scope"]: row for row in result14.collect()}
assert result14_dict["row_count_Dataset A"]["count_value"] == 100
assert result14_dict["row_count_Dataset B"]["count_value"] == 100
assert result14_dict["common_rows_both"]["count_value"] == 99
assert result14_dict["union_rows_both"]["count_value"] == 101  # 100 + 100 - 99
jaccard_pct = result14_dict["jaccard_pct_both"]["percent_value"]
expected_jaccard = 99 / 101 * 100  # ~98.0198%
assert abs(jaccard_pct - expected_jaccard) < 0.01, f"Expected ~{expected_jaccard}, got {jaccard_pct}"
print(f"✓ TEST 14 PASSED: Decimal precision verified (Jaccard: {jaccard_pct}%)")


[TEST 14] Decimal precision in percentage calculations
+--------------+-------------------+-----------+-------------+
|metric        |scope              |count_value|percent_value|
+--------------+-------------------+-----------+-------------+
|row_count     |Dataset A          |100        |NULL         |
|row_count     |Dataset B          |100        |NULL         |
|diff_rows     |Dataset A          |1          |NULL         |
|diff_rows     |Dataset B          |1          |NULL         |
|common_rows   |both               |99         |NULL         |
|union_rows    |both               |101        |NULL         |
|match_pct     |Dataset A→Dataset B|99         |99.0         |
|match_pct     |Dataset B→Dataset A|99         |99.0         |
|jaccard_pct   |both               |99         |98.0198      |
|size_ratio_pct|Dataset A→Dataset B|100        |100.0        |
+--------------+-------------------+-----------+-------------+

✓ TEST 14 PASSED: Decimal precision verified (Jaccard: 98.019